# Run Polaris on 86 FOVs in LM50GS Brain TISSUE dataset
- 0.85 spot detection probability
- GS registration method
- *No* butter2D filter
- Added chromatic correction from GS

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.utils import get_file

from deepcell.utils.plot_utils import create_rgb_image
from deepcell.datasets import SpotNetExampleData
from deepcell_spots.applications import Polaris
from deepcell_spots.utils.results_utils import mask_spots

# ---- Main folder = the repository folder, which holds Data/ (layout described in Data/README.md). ----
# Run this from the folder the file is in, or set the environment variable GUIDESTAR_MAIN_DIR to the repository path.
main_dir = os.environ.get('GUIDESTAR_MAIN_DIR', '../..')


os.environ["DEEPCELL_ACCESS_TOKEN"] = "<your DeepCell access token>" # Set Deepcell access token

<user_home>\miniforge3\envs\Polaris38\lib\site-packages\keras\optimizer_v2\gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  # Restrict TensorFlow to only allocate 1GB of memory on the first GPU
  try:
    tf.config.set_logical_device_configuration(
        gpus[0],
        [tf.config.LogicalDeviceConfiguration(memory_limit=5000)])
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Virtual devices must be set before GPUs have been initialized
    print(e)

1 Physical GPUs, 1 Logical GPUs


In [3]:
import pandas as pd
# load tiff files for triton format 
data_path = os.path.join(main_dir, 'Data/20250728_Lm50GS_mbrain_3cT3/')
#os.path.join(main_dir, 'Data/100uM/')
hyb_list = [1, 1, 2, 3, 4, 2, 3, 5, 1, 2, 4, 5, 6, 3, 4, 5] #[9,8,7,6,5,4,2,3,9,8,7,6,5,4,3,2]
# type_list -> channel list 
type_list = ['Cy5', 'Cy7', 'Cy7', 'Cy7', 'Cy7', 'Cy5', 'Cy5', 'Cy7', 'Alexa594', 'Alexa594', 'Cy5', 'Cy5', 'Cy5', 'Alexa594', 'Alexa594', 'Alexa594']
channel_hyb = [f"{t}_{h:02d}" for t, h in zip(type_list, hyb_list)]
rounds = 6
channels = 3

df_barcodes = pd.read_csv(os.path.join(main_dir, 'Data/20250728_Lm50GS_mbrain_3cT3/codebook/codebook_data.tsv'),
                          header=None,
                          sep="\t")

#modify the barcode to fit sample format
last_col = df_barcodes.columns[-1]
df_barcodes = df_barcodes[[last_col] + list(df_barcodes.columns[:-1])]
df_barcodes.columns = ['Gene'] + channel_hyb
df_barcodes

,Gene,Cy5_01,Cy7_01,Cy7_02,Cy7_03,Cy7_04,Cy5_02,Cy5_03,Cy7_05,Alexa594_01,Alexa594_02,Cy5_04,Cy5_05,Cy5_06,Alexa594_03,Alexa594_04,Alexa594_05
0,Klf6,0,0,1,1,1,0,0,0,0,0,0,0,1,0,0,0
1,Fosb,1,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0
2,Slc1a3,0,0,1,1,0,0,0,1,0,0,1,0,0,0,0,0
3,Hcn1,0,0,1,1,0,1,0,0,0,1,0,0,0,0,0,0
4,Sod2,0,0,1,0,0,1,0,0,0,0,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,Blank-20,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,1
136,Blank-21,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0
137,Blank-22,0,0,0,0,0,0,1,1,0,1,0,0,0,1,0,0
138,Blank-23,0,1,0,0,0,0,0,1,0,0,1,1,0,0,0,0


In [4]:
multiplex_app = Polaris(image_type='multiplex', 
                        segmentation_type='no segmentation',
                        decoding_kwargs={'rounds': 16, 'channels': 1,
                                        'df_barcodes': df_barcodes})

INFO:root:Checking for cached data
INFO:root:Checking SpotDetection-8.tar.gz against provided file_hash...
INFO:root:SpotDetection-8.tar.gz with hash a6164e48ef8872a9524b4ec6726859d7 already available.
INFO:root:Extracting <user_home>\.deepcell\models\SpotDetection-8.tar.gz
INFO:root:Successfully extracted <user_home>\.deepcell\models\SpotDetection-8.tar.gz into <user_home>\.deepcell\models
<user_home>\miniforge3\envs\Polaris38\lib\site-packages\deepcell_spots\applications\polaris.py:201: UserWarning: No segmentation application instantiated.
  warnings.warn('No segmentation application instantiated.')


Load related functions below

In [5]:
def _upsampled_dft(data, upsampled_region_size,
                   upsample_factor=1, axis_offsets=None):
    """
    Upsampled DFT by matrix multiplication.
    This code is intended to provide the same result as if the following
    operations were performed:
        - Embed the array "data" in an array that is ``upsample_factor`` times
          larger in each dimension.  ifftshift to bring the center of the
          image to (1,1).
        - Take the FFT of the larger array.
        - Extract an ``[upsampled_region_size]`` region of the result, starting
          with the ``[axis_offsets+1]`` element.
    It achieves this result by computing the DFT in the output array without
    the need to zeropad. Much faster and memory efficient than the zero-padded
    FFT approach if ``upsampled_region_size`` is much smaller than
    ``data.size * upsample_factor``.
    Parameters
    ----------
    data : array
        The input data array (DFT of original data) to upsample.
    upsampled_region_size : integer or tuple of integers, optional
        The size of the region to be sampled.  If one integer is provided, it
        is duplicated up to the dimensionality of ``data``.
    upsample_factor : integer, optional
        The upsampling factor.  Defaults to 1.
    axis_offsets : tuple of integers, optional
        The offsets of the region to be sampled.  Defaults to None (uses
        image center)
    Returns
    -------
    output : ndarray
            The upsampled DFT of the specified region.
    """
    # if people pass in an integer, expand it to a list of equal-sized sections
    if not hasattr(upsampled_region_size, "__iter__"):
        upsampled_region_size = [upsampled_region_size, ] * data.ndim
    else:
        if len(upsampled_region_size) != data.ndim:
            raise ValueError("shape of upsampled region sizes must be equal "
                             "to input data's number of dimensions.")

    if axis_offsets is None:
        axis_offsets = [0, ] * data.ndim
    else:
        if len(axis_offsets) != data.ndim:
            raise ValueError("number of axis offsets must be equal to input "
                             "data's number of dimensions.")

    im2pi = 1j * 2 * np.pi

    dim_properties = list(zip(data.shape, upsampled_region_size, axis_offsets))

    for (n_items, ups_size, ax_offset) in dim_properties[::-1]:
        kernel = ((np.arange(ups_size) - ax_offset)[:, None]
                  * np.fft.fftfreq(n_items, upsample_factor))
        kernel = np.exp(-im2pi * kernel)

        # Equivalent to:
        #   data[i, j, k] = kernel[i, :] @ data[j, k].T
        data = np.tensordot(kernel, data, axes=(1, -1))
    return data



def register_translation(src_image: np.ndarray,
                         target_image: np.ndarray,
                         upsample_factor: int = 1,
                         space: str = "real",
                         return_error: bool = True,
                         ):
    """
    Efficient subpixel image translation registration by cross-correlation.
    This code gives the same precision as the FFT upsampled cross-correlation
    in a fraction of the computation time and with reduced memory requirements.
    It obtains an initial estimate of the cross-correlation peak by an FFT and
    then refines the shift estimation by upsampling the DFT only in a small
    neighborhood of that estimate by means of a matrix-multiply DFT.
    Parameters
    ----------
    src_image : array
        Reference image.
    target_image : array
        Image to register.  Must be same dimensionality as ``src_image``.
    upsample_factor : int, optional
        Upsampling factor. Images will be registered to within
        ``1 / upsample_factor`` of a pixel. For example
        ``upsample_factor == 20`` means the images will be registered
        within 1/20th of a pixel.  Default is 1 (no upsampling)
    space : string, one of "real" or "fourier", optional
        Defines how the algorithm interprets input data.  "real" means data
        will be FFT'd to compute the correlation, while "fourier" data will
        bypass FFT of input data.  Case insensitive.
    return_error : bool, optional
        Returns error and phase difference if on,
        otherwise only shifts are returned
    Returns
    -------
    shifts : ndarray
        Shift vector (in pixels) required to register ``target_image`` with
        ``src_image``.  Axis ordering is consistent with numpy (e.g. Z, Y, X)
    error : float
        Translation invariant normalized RMS error between ``src_image`` and
        ``target_image``.
    phasediff : float
        Global phase difference between the two images (should be
        zero if images are non-negative).
    References
    ----------
    .. [1] Manuel Guizar-Sicairos, Samuel T. Thurman, and James R. Fienup,
           "Efficient subpixel image registration algorithms,"
           Optics Letters 33, 156-158 (2008). :DOI:`10.1364/OL.33.000156`
    .. [2] James R. Fienup, "Invariant error metrics for image reconstruction"
           Optics Letters 36, 8352-8357 (1997). :DOI:`10.1364/AO.36.008352`
    """
    # images must be the same shape
    if src_image.shape != target_image.shape:
        raise ValueError("Error: images must be same size for "
                         "register_translation")

    # assume complex data is already in Fourier space
    if space.lower() == 'fourier':
        src_freq = src_image
        target_freq = target_image
    # real data needs to be fft'd.
    elif space.lower() == 'real':
        src_freq = np.fft.fftn(src_image)
        target_freq = np.fft.fftn(target_image)
    else:
        raise ValueError("Error: register_translation only knows the \"real\" "
                         "and \"fourier\" values for the ``space`` argument.")

    # Whole-pixel shift - Compute cross-correlation by an IFFT
    shape = src_freq.shape
    image_product = src_freq * target_freq.conj()
    cross_correlation = np.fft.ifftn(image_product)

    # Locate maximum
    maxima = np.unravel_index(np.argmax(np.abs(cross_correlation)),
                              cross_correlation.shape)
    midpoints = np.array([np.fix(axis_size / 2) for axis_size in shape])

    shifts = np.array(maxima, dtype=np.float64)
    shifts[shifts > midpoints] -= np.array(shape)[shifts > midpoints]

    # calculate the pixel-resolution maximum cross-correlation value
    CCmax_pixel = cross_correlation[maxima]

    if upsample_factor == 1:
        if return_error:
            src_amp = np.sum(np.abs(src_freq) ** 2) / src_freq.size
            target_amp = np.sum(np.abs(target_freq) ** 2) / target_freq.size
            CCmax = CCmax_pixel # this is height of cc peak

    # If upsampling > 1, then refine estimate with matrix multiply DFT
    else:
        # Initial shift estimate in upsampled grid
        shifts = np.round(shifts * upsample_factor) / upsample_factor
        upsampled_region_size = np.ceil(upsample_factor * 1.5)

        # Center of output array at dftshift + 1
        dftshift = np.fix(upsampled_region_size / 2.0)
        upsample_factor = np.array(upsample_factor, dtype=np.float64)
        normalization = (src_freq.size * upsample_factor ** 2)

        # Matrix multiply DFT around the current shift estimate
        sample_region_offset = dftshift - shifts * upsample_factor
        cross_correlation = _upsampled_dft(image_product.conj(),
                                           upsampled_region_size,
                                           upsample_factor,
                                           sample_region_offset).conj()
        cross_correlation /= normalization

        # Locate maximum and map back to original pixel grid
        maxima = np.unravel_index(np.argmax(np.abs(cross_correlation)),
                                  cross_correlation.shape)
        CCmax = cross_correlation[maxima]

        maxima = np.array(maxima, dtype=np.float64) - dftshift

        shifts = shifts + maxima / upsample_factor

        if return_error:
            src_amp = _upsampled_dft(src_freq * src_freq.conj(),
                                     1, upsample_factor)[0, 0]
            src_amp /= normalization
            target_amp = _upsampled_dft(target_freq * target_freq.conj(),
                                        1, upsample_factor)[0, 0]
            target_amp /= normalization

    # For singleton dimensions, the shift calculated has no effect.
    # Set to zero.
    for dim in range(src_freq.ndim):
        if shape[dim] == 1:
            shifts[dim] = 0

    if return_error:
        # return (shifts,
        #         _compute_error(CCmax, src_amp, target_amp),
        #         _compute_phasediff(CCmax)
        #         )

        return (shifts,
                np.abs(CCmax),
                np.abs(CCmax_pixel)
                )
    else:
        return shifts

In [7]:
from skimage.transform import AffineTransform
def parseDaybook(filepath: str,
                 verbose: bool = True,
                 ) -> AffineTransform:
    """
    parse daybook lateral chromatic shift output file
    and generate affine matrix

    returns affine matrix ready for warping with skimage
    """
    title_row = None
    values_row = None

    with open(filepath) as csv_file:

        csv_reader = csv.reader(csv_file, delimiter=';')

        for row in csv_reader:
            if "a_affine_parameter" in row:
                title_row = [title.strip() for title in row]
            # go to next row. then stop reading
            elif title_row is not None:
                values_row = [float(value) for value in row]
                break

    assert title_row is not None, "title row for param values not found in daybook csv file"
    assert values_row is not None, "row containing param values not found in daybook csv file"

    matrix = np.zeros((3, 3), dtype=np.float64)
    matrix[0, 0] = values_row[title_row.index("a_affine_parameter")]
    matrix[0, 1] = values_row[title_row.index("b_affine_parameter")]
    matrix[1, 0] = values_row[title_row.index("c_affine_parameter")]
    matrix[1, 1] = values_row[title_row.index("d_affine_parameter")]
    matrix[0, 2] = values_row[title_row.index("x_translation_affine_parameter")]
    matrix[1, 2] = values_row[title_row.index("y_translation_affine_parameter")]
    matrix[2, 2] = 1

    if verbose:
        print("____ Matrix: ____\n", matrix)

    return AffineTransform(matrix)

In [8]:
import json
import tifffile
def read_tif_GS(Guidestar_img_name, metadata_filepath, chan_hyb):
    with open(metadata_filepath, 'r') as metadata_file:
        meta_data = json.load(metadata_file)
    # list of colours/channels corresponding to the frames of the tiff
    colour_list = meta_data["Summary"]["ChNames"]
    channel_index = colour_list.index(chan_hyb.split('_')[0])
    tiff_img = tifffile.imread(Guidestar_img_name, is_ome=False, is_mmstack=False) # somehow in polaris38 environment, its (all fovs, channel, 2048,2048)
    
    smallest_dimension = np.argmin(tiff_img.shape)
    if smallest_dimension == 2:  # last dim is frames
        Guidestar_img = tiff_img[..., channel_index]

    elif smallest_dimension == 0:  # first dim is frames
        Guidestar_img = tiff_img[channel_index, ...]
    return Guidestar_img

## ALL FOV prediction

In [11]:
import re
from skimage import transform
import copy
from skimage.transform import warp as skimage_warp
import skimage
from scipy import ndimage
import csv

freq_filter = None # drop butter2d
reference_bit = 0 # bit for image registration/alignment
upsampling = 50
normalization = None
verbose = True
type_to_colour = {
    "Cy3": "Cy3",
    "Alexa594": "Alexa594",
    "Cy5": "Cy5",
    "Cy7": "Cy7"
}
reference_colour = "Cy5"
calibration_path = os.path.join(main_dir, 'Data/20250728_Lm50GS_mbrain_3cT3/calibrationTriton')
color_list = type_list
dropped_bits = []
save_path = os.path.join(main_dir, 'Data/20250728_Lm50GS_mbrain_3cT3/output/') + str(multiplex_app.distribution) + '_thresh0.85_GS_chrco_img_regi_no_butter2d'
os.makedirs(save_path, exist_ok=True)
save_regi_output = os.path.join(save_path, f'shift_registration_polaris_splitfish_method_lm50gs_nobutter2d.txt')
os.makedirs(os.path.dirname(save_regi_output), exist_ok=True)

params_pattern1 = re.compile("[a-zA-Z]+(\d+)_[a-zA-Z]+(\d+)")
params_pattern2 = re.compile("([a-zA-Z0-9]+)to([a-zA-Z0-9]+)")

pairs_tf_dict = {}

for file in os.listdir(calibration_path):

    if os.path.splitext(file)[-1] == ".csv":

        match1 = re.search(params_pattern1, file)
        match2 = re.search(params_pattern2, file)

        if match1:
            match = match1
        elif match2:
            match = match2
        else:
            continue

        daybook_fullpath = os.path.join(calibration_path, file)
        colour_pair = (match.group(1), match.group(2))

        pairs_tf_dict[colour_pair] = parseDaybook(daybook_fullpath)
colours_tf_dict = {reference_colour: None}
for pair in pairs_tf_dict:
    if pair[0] == reference_colour:
        colours_tf_dict[pair[1]] = pairs_tf_dict[pair]
    if pair[1] == reference_colour:
        colours_tf_dict[pair[0]] = pairs_tf_dict[pair].inverse

# 86 selected fovs for Lm50GS
fov_list = [(0, 4),(0, 5),(0, 6),(0, 7),(0, 8),(0, 9),
 (1, 1),(1, 2),(1, 3),(1, 4),(1, 5),(1, 6),(1, 7), (1, 8),(1, 9),
 (2, 0),(2, 1),(2, 2),(2, 3),(2, 5),(2, 6),(2, 7),(2, 8),(2, 9),
 (3, 0), (3, 1), (3, 2), (3, 3), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9),
 (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 7), (4, 8), (4, 9),
 (5, 0), (5, 1), (5, 3), (5, 4), (5, 5), (5, 7), (5, 8), (5, 9),
 (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6), (6, 7), (6, 8), (6, 9),
 (7, 0), (7, 1), (7, 2), (7, 3), (7, 4), (7, 6), (7, 7), (7, 8),
 (8, 1), (8, 2), (8, 3), (8, 5), (8, 6), (8, 7), (8, 8), (8, 9),
 (9, 0), (9, 1), (9, 2), (9, 3), (9, 4), (9, 5), (9, 6), (9, 7), (9, 8), (9, 9)]
fov_list = [f"{x:03d}_{y:03d}" for x, y in fov_list]

## full fov for indexing because this version of tiff read takes in all 100 fovs
full_list = [(xpos,ypos) for xpos in range(10) for ypos in range(10)]
full_list = [f"{x:03d}_{y:03d}" for x, y in full_list]


## loop through each FOV
for fov in fov_list:
    print('processing fov: ' + str(fov))
    tif_image = np.zeros((2048,2048,16)) # for registration
    final_tif_img = np.zeros((1,2048,2048,16))
    # Load 16 bits array into dax_regi dictionary, key correspond to respective bits for registration
    #dax_regi = {} #np.zeros((1,2048,2048,16))
    for i, info in enumerate(channel_hyb): #because there are 12 hybs 2 channels 
        tif_name = 'S10/hyb' + f"{hyb_list[i]:02d}" + '_1/hyb'+ f"{hyb_list[i]:02d}" + f'_1_MMStack_1-Pos_{fov}.ome.tif' # str(info) + '_' + f"{fov:02d}" + '.dax'
        meta_name = 'S10/hyb' + f"{hyb_list[i]:02d}" + '_1/hyb'+ f"{hyb_list[i]:02d}" + f'_1_MMStack_1-Pos_{fov}_metadata.txt'
        print('Processing ' + str(data_path + tif_name))
        tif_image[...,i] = read_tif_GS(str(data_path + tif_name), str(data_path + meta_name), channel_hyb[i])
        print(f'image shape {tif_image[...,i].shape}')
        if i == reference_bit:
            reference_image = copy.deepcopy(tif_image[...,i])
        #dax_regi[i] = np.expand_dims(image_data, axis=[0, -1]) #from 2048*2048 to 1*2048*2048*1

    ### insert Guidestar chromatic distortion correction ###
    for bit, colour in enumerate(color_list):
        colour = color_list[bit]
        print(
            f"Correcting chromatic distortion for bit {bit} with colour {colour}...\n"
        )

        if bit not in dropped_bits:
            transform = colours_tf_dict[type_to_colour[colour]]
            if transform is not None:
                slice_to_warp = tif_image[...,bit]
                align_slice = skimage_warp(slice_to_warp, transform)
                align_slice = np.expand_dims(align_slice, axis=0)
                tif_image[..., bit] = align_slice
    
    ### insert starfish registration ###
    ### modified to eliminate imagestack data structure ###
    
    #prepare reference img
    ref_slice = copy.copy(tif_image[..., reference_bit])
    ref_slice_fourier = np.fft.fftn(ref_slice)
    

    for bit in range(len(channel_hyb)):
        target_image = tif_image[..., bit]
        if len(target_image.shape) != 2:
            raise ValueError(
                f"Not 2D image! "
            )
        current_slice = copy.copy(target_image)
        current_slice_fourier = np.fft.fftn(current_slice)


        ### prepare reference bit
        if bit == reference_bit:
            if freq_filter is not None:
                ref_filtered_slice = np.fft.ifftn(ref_slice_fourier * filter_shifted) #fourier filter_shift 
                final_tif_img[..., reference_bit] = ref_filtered_slice.real #save the filtered shift result
            else:
                final_tif_img[..., reference_bit] = tif_image[..., reference_bit]
            continue

        (shifts,
        fine_error,
        pixel_error) = register_translation(ref_slice_fourier,
                                            current_slice_fourier,
                                            upsample_factor=upsampling,
                                            space="fourier")
        with open(save_regi_output, 'a') as f: # record shift result for debugging 
            formatted_shifts = ', '.join([f'{x:.3f}' for x in shifts])
            f.write(f"Bit {bit} in fov: {fov} shift merfish by {formatted_shifts}\n")
        ## align image after registration
        registered_slice = np.fft.ifftn(
                        ndimage.fourier_shift(current_slice_fourier, shifts)
                    )
        final_img = registered_slice.real
        
        if freq_filter is not None:
                registered_filtered_slice = np.fft.ifftn(
                    ndimage.fourier_shift(current_slice_fourier * filter_shifted,
                                                shifts)
                        )
                final_img = registered_filtered_slice.real
        final_tif_img[..., bit] = final_img
    #####################################
    
    background_image = np.max(final_tif_img, axis=-1, keepdims=True)
    mask = mask_spots(background_image, 0.99)

    multiplex_pred = multiplex_app.predict(spots_image=final_tif_img,
                                            #segmentation_image=mesmer_image,
                                            mask=mask,
                                            image_mpp=0.11, clip=True, 
                                            threshold=0.85) #Probability threshold for a pixel to be decoded
    df_results = multiplex_pred[0]
    segmentation_result = multiplex_pred[1]
    df_results.to_csv(save_path + '/' + str(fov) + '_Polaris_prediction.csv')



____ Matrix: ____
 [[ 1.00126148e+00  3.97436125e-05 -2.18404367e+00]
 [-3.97436125e-05  1.00126148e+00 -1.11771017e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
____ Matrix: ____
 [[ 1.00196723e+00  4.26314977e-05 -3.17444090e+00]
 [-4.26314977e-05  1.00196723e+00 -2.59335733e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
____ Matrix: ____
 [[ 9.97952527e-01  5.40682540e-07  2.15458376e+00]
 [-5.40682540e-07  9.97952527e-01  2.22519960e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
processing fov: 003_000
Processing <main_dir>/Data/20250728_Lm50GS_mbrain_3cT3/S10/hyb01_1/hyb01_1_MMStack_1-Pos_003_000.ome.tif
image shape (2048, 2048)
Processing <main_dir>/Data/20250728_Lm50GS_mbrain_3cT3/S10/hyb01_1/hyb01_1_MMStack_1-Pos_003_000.ome.tif
image shape (2048, 2048)
Processing <main_dir>/Data/20250728_Lm50GS_mbrain_3cT3/S10/hyb02_1/hyb02_1_MMStack_1-Pos_003_000.ome.tif
image shape (2048, 2048)
Processing <main_dir>/Data/20250728_Lm50GS_mbrain_3cT3/S10/hyb03_1/

100%|██████████| 16/16 [2:19:34<00:00, 523.40s/it] 


Decoding gene identities.
Training...


100%|██████████| 500/500 [00:16<00:00, 29.98it/s]


Estimating barcode probabilities...
Revising errors...


188727it [05:50, 538.71it/s] 


... (363 outputs left out)

100%|██████████| 500/500 [00:18<00:00, 27.67it/s]


Estimating barcode probabilities...
Revising errors...


191152it [04:35, 694.48it/s] 


63530 of 92840 rescue attempts were successful.
Refining spot locations.
